In [1]:
import numpy as np

# 激活函数及其导数
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    # Sigmoid 的导数正好等于：它自己的输出 * (1 - 它自己的输出)
    return x * (1 - x)

# 1. 模拟一些简单的训练数据 (X 包含 2 个特征，Y 是标签 0 或 1)
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
Y = np.array([[0], [1], [1], [0]]) # 经典的异或(XOR)问题，线性无法切分

# 2. 随机初始化 W 和 b (严格按照你的步骤)
np.random.seed(42)
# 输入层到隐藏层 (2个输入 -> 3个隐藏神经元)
W1 = np.random.randn(2, 3)
b1 = np.zeros((1, 3))
# 隐藏层到输出层 (3个隐藏神经元 -> 1个输出)
W2 = np.random.randn(3, 1)
b2 = np.zeros((1, 1))

lr = 0.1 # 学习率

# 3. 开启训练循环 (反复这一过程)
for epoch in range(10000):
    # ---- 【前向传播】 ----
    # 隐藏层计算
    z1 = np.dot(X, W1) + b1
    a1 = sigmoid(z1)

    # 输出层计算
    z2 = np.dot(a1, W2) + b2
    a2 = sigmoid(z2) # a2 就是最终求出的 y_predict

    # 计算损失 (你提到的：减去输入的 Y 并平方再乘以二分之一)
    loss = 0.5 * np.mean((a2 - Y) ** 2)

    # ---- 【反向传播：链式求导】 ----
    # 第一步：求最后一层的误差项 (Loss对z2的偏导)
    # dLoss/da2 = (a2 - Y), da2/dz2 = a2 * (1 - a2)
    delta2 = (a2 - Y) * sigmoid_derivative(a2)

    # 第二步：最后一层参数的梯度
    dW2 = np.dot(a1.T, delta2)
    db2 = np.sum(delta2, axis=0, keepdims=True)

    # 第三步：把误差往前传，求隐藏层的误差项 (Loss对z1的偏导)
    # 通过 W2 把输出层的误差拉回来，再乘以隐藏层自己的激活函数导数
    delta1 = np.dot(delta2, W2.T) * sigmoid_derivative(a1)

    # 第四步：第一层参数的梯度
    dW1 = np.dot(X.T, delta1)
    db1 = np.sum(delta1, axis=0, keepdims=True)

    # ---- 【更新 W 和 b】 ----
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

    if epoch % 2000 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

print("\n手写版本最终预测结果:")
print(a2)

Epoch 0, Loss: 0.1591
Epoch 2000, Loss: 0.0709
Epoch 4000, Loss: 0.0101
Epoch 6000, Loss: 0.0031
Epoch 8000, Loss: 0.0017

手写版本最终预测结果:
[[0.02515564]
 [0.95263635]
 [0.95122343]
 [0.0627247 ]]


In [3]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 加载真实的红酒数据集
wine = load_wine()
# 选取前4个实用理化特征：酒精含量(Alcohol)、苹果酸(Malic acid)、灰分(Ash)、碱度(Alcalinity of ash)
X_raw = wine.data[:, :4]
# 原始数据有3类，我们改成二分类：高质量酒(标签1)和普通酒(标签0)
y_raw = (wine.target == 0).astype(int).reshape(-1, 1)

# 2. 划分训练集与测试集 (实用场景必做)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.3, random_state=42)

# 3. 特征标准化 (真实多特征数据如果不做标准化，网络绝不收敛！)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# 4. 动态适配网络维度 (4个输入特征 -> 5个隐藏层神经元 -> 1个输出)
input_dim = X_train.shape[1]  # 值为 4
hidden_dim = 5
output_dim = 1

# 随机初始化 W 和 b
np.random.seed(42)
W1 = np.random.randn(input_dim, hidden_dim) * 0.1
b1 = np.zeros((1, hidden_dim))
W2 = np.random.randn(hidden_dim, output_dim) * 0.1
b2 = np.zeros((1, output_dim))

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500))) # clip防止数据过大溢出

def sigmoid_derivative(x):
    return x * (1 - x)

lr = 0.05
epochs = 5000

# 5. 训练网络
for epoch in range(epochs):
    # 【前向传播】
    z1 = np.dot(X_train, W1) + b1
    a1 = sigmoid(z1)
    z2 = np.dot(a1, W2) + b2
    a2 = sigmoid(z2)

    # 【反向传播】
    delta2 = (a2 - y_train) * sigmoid_derivative(a2)
    dW2 = np.dot(a1.T, delta2) / X_train.shape[0] # 除以样本数，防止样本量大时梯度爆炸
    db2 = np.sum(delta2, axis=0, keepdims=True) / X_train.shape[0]

    delta1 = np.dot(delta2, W2.T) * sigmoid_derivative(a1)
    dW1 = np.dot(X_train.T, delta1) / X_train.shape[0]
    db1 = np.sum(delta1, axis=0, keepdims=True) / X_train.shape[0]

    # 【更新参数】
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

# 6. 在测试集上检验真本事 (用没见过的数据验证)
z1_test = np.dot(X_test, W1) + b1
a1_test = sigmoid(z1_test)
z2_test = np.dot(a1_test, W2) + b2
y_pred_prob = sigmoid(z2_test)
y_pred = (y_pred_prob > 0.5).astype(int)

accuracy = np.mean(y_pred == y_test)
print(f"NumPy 手写版本在红酒测试集上的准确率: {accuracy * 100:.2f}%")

NumPy 手写版本在红酒测试集上的准确率: 94.44%


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 准备数据 (PyTorch 需要转换为张量 Tensor)
X_tensor = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
Y_tensor = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

# 2. 搭建网络结构
class PyTorchBPNN(nn.Module):
    def __init__(self):
        super(PyTorchBPNN, self).__init__()
        # 对应手写版的 W1, b1
        self.hidden = nn.Linear(2, 3)
        # 对应手写版的 W2, b2
        self.output = nn.Linear(3, 1)
        # 激活函数
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 严格执行：线性 -> 激活 -> 线性 -> 激活
        x = self.sigmoid(self.hidden(x))
        x = self.sigmoid(self.output(x))
        return x

# 实例化网络
model = PyTorchBPNN()

# 定义损失函数 (MSE 均方误差，就是你说的平方乘以二分之一)
criterion = nn.MSELoss()

# 定义优化算法 (标准的梯度下降 SGD)
optimizer = optim.SGD(model.parameters(), lr=0.1)

# 3. 训练循环
for epoch in range(10000):
    # 前向传播求出 y
    predictions = model(X_tensor)

    # 计算 Loss
    loss = criterion(predictions, Y_tensor)

    # 反向传播 (PyTorch 自动为你做完刚才手写的所有链式求导！)
    optimizer.zero_grad() # 清空上一轮的梯度
    loss.backward()        # 自动计算梯度
    optimizer.step()       # 自动更新 W 和 b

    if epoch % 2000 == 0:
        print(f"PyTorch Epoch {epoch}, Loss: {loss.item():.4f}")

print("\nPyTorch版本最终预测结果:")
with torch.no_grad():
    print(model(X_tensor))

PyTorch Epoch 0, Loss: 0.2577
PyTorch Epoch 2000, Loss: 0.2492
PyTorch Epoch 4000, Loss: 0.2461
PyTorch Epoch 6000, Loss: 0.2211
PyTorch Epoch 8000, Loss: 0.1248

PyTorch版本最终预测结果:
tensor([[0.1462],
        [0.8258],
        [0.8258],
        [0.2228]])


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 转换数据为 PyTorch 张量
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test)

# 2. 构建生产级网络
class WinePredictor(nn.Module):
    def __init__(self, in_features):
        super(WinePredictor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 5),
            nn.Sigmoid(),
            nn.Linear(5, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = WinePredictor(in_features=input_dim)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.05)

# 3. 训练循环
for epoch in range(5000):
    predictions = model(X_train_t)
    loss = criterion(predictions, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 4. 评估测试集
model.eval() # 切换到评估模式
with torch.no_grad():
    test_preds_prob = model(X_test_t)
    test_preds = (test_preds_prob > 0.5).float()
    torch_accuracy = (test_preds == y_test_t).float().mean()

print(f"PyTorch 调包版本在红酒测试集上的准确率: {torch_accuracy.item() * 100:.2f}%")

PyTorch 调包版本在红酒测试集上的准确率: 92.59%
